# Homework 2: PyTorch Basics

**Assignment total: 5 points. Question 4: 2.00 points.**

This notebook contains the scaffold for **Question 4 only**. Complete the TODO code cells and run all cells in order.

For **Questions 1–3**, write your own code without a supplied template. Put your written answers and screenshots of execution results (1.3, 2 and 3) in the PDF report. The report has no page limit. Keep the code you wrote so your results can be reproduced.


## Setup (provided)

Run from the HW2 directory. CPU is sufficient. Install a compatible stable torch/torchvision pair using the [official selector](https://pytorch.org/get-started/locally/) and install Jupyter. Keep the seed and prescribed configuration fixed: batch size 64, SGD learning rate 0.1, and 5 epochs.

The local MNIST files are included in the student package. Teacher solutions are separate.


In [ ]:
import os
from pathlib import Path
if not Path("prepare_mnist.py").is_file() and Path("../prepare_mnist.py").is_file():
    os.chdir("..")
import sys
sys.path.insert(0, str(Path.cwd()))
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
import torchvision
from prepare_mnist import prepare_mnist

torch.manual_seed(2026)
torch.set_num_threads(2)
print("Python:", sys.version.split()[0])
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
print("device: cpu")


## 4. MNIST classification with a two-layer MLP (2.00 points)

Build the full workflow: load data → define the model → calculate cross-entropy → backpropagate and train → test → save and reload.

Use all 60,000 official training examples for training and all 10,000 official test examples for the final evaluation. Evaluate the test set after 5 epochs; do not use test results to choose settings or checkpoints.

### 4(a). Data loading and preprocessing (0.50)

Use `v2.ToImage()`, `v2.ToDtype(torch.float32, scale=True)` and `v2.Normalize((0.5,), (0.5,))`. Apply the same transform to both splits. Return `(train_loader, test_loader)` with batch size 64 and `num_workers=0`; shuffle only the training loader. Set its generator to `torch.Generator().manual_seed(2026)`.


In [ ]:
def make_mnist_loaders(root="data"):
    available = prepare_mnist(root=root)
    # TODO: define the transform and both MNIST datasets.
    # Use train=True / train=False and download=not available.
    # TODO: create and return (train_loader, test_loader).
    raise NotImplementedError("Complete make_mnist_loaders")


#### Data self-check (provided)

Record the batch properties in your report. Explain Dataset versus DataLoader and why `num_workers=0` loads data in the main process.


In [ ]:
train_loader, test_loader = make_mnist_loaders()
assert len(train_loader.dataset) == 60000 and len(test_loader.dataset) == 10000
assert train_loader.dataset.train and not test_loader.dataset.train
assert train_loader.batch_size == test_loader.batch_size == 64
assert train_loader.num_workers == test_loader.num_workers == 0
assert isinstance(train_loader.sampler, torch.utils.data.RandomSampler)
assert isinstance(test_loader.sampler, torch.utils.data.SequentialSampler)
images, labels = next(iter(train_loader))
assert images.shape == (64, 1, 28, 28) and labels.shape == (64,)
assert images.dtype == torch.float32 and labels.dtype == torch.int64
assert images.min() >= -1 and images.max() <= 1
assert labels.min() >= 0 and labels.max() <= 9
print("Training/test examples:", len(train_loader.dataset), len(test_loader.dataset))
print("Images:", tuple(images.shape), images.dtype)
print("Labels:", tuple(labels.shape), labels.dtype)
print("Image range:", images.min().item(), images.max().item())


### 4(b). Model definition (0.25)

Implement `Flatten → Linear(784, 128) → ReLU → Linear(128, 10)` in an `nn.Module`. Two layers means **two Linear layers and one hidden layer**. Return raw logits of shape `(B, 10)`; do not apply softmax.


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: register flattening, two Linear layers and ReLU.
        raise NotImplementedError("Complete MLP.__init__")

    def forward(self, x):
        # TODO: flatten each image and return class logits.
        raise NotImplementedError("Complete MLP.forward")


In [ ]:
torch.manual_seed(2026)
model = MLP()
linear_layers = [m for m in model.modules() if isinstance(m, nn.Linear)]
assert [(m.in_features, m.out_features) for m in linear_layers] == [(784, 128), (128, 10)]
with torch.no_grad():
    logits = model(images[:7])
assert logits.shape == (7, 10) and torch.isfinite(logits).all()
print(model)
print("Logit shape:", tuple(logits.shape))


### 4(c). Loss, backpropagation and training (0.50)

Return `nn.CrossEntropyLoss()` and `torch.optim.SGD(model.parameters(), lr=0.1)` from `make_training_components`.

In `train_one_epoch`, use training mode. For each batch: clear gradients → compute logits → calculate loss → call backward → update parameters. Accumulate `loss.item() * batch_size` and the number of correct predictions from these logits. Return `(mean_loss, accuracy)`, where accuracy is a fraction in `[0, 1]`, and both averages use the total number of examples.


In [ ]:
def make_training_components(model):
    # TODO: create and return (loss_fn, optimizer).
    raise NotImplementedError("Complete make_training_components")


In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    # TODO: set training mode and initialize metric accumulators.
    # TODO: iterate over batches; calculate loss, backpropagate and update.
    # TODO: return sample-weighted mean loss and accuracy.
    raise NotImplementedError("Complete train_one_epoch")


### 4(d). Evaluation and testing (0.25)

Implement `evaluate` using `model.eval()` and `torch.no_grad()`. Return `(mean_loss, accuracy)` over **all** examples. Include the smaller final batch and weight its loss by its number of examples. No parameters should be updated.

The supplied runner uses this function to measure full-training-set loss before and after training, and then evaluates the test set once. Complete both training and evaluation functions before running it.


In [ ]:
def evaluate(model, loader, loss_fn):
    # TODO: switch to evaluation mode and disable gradient recording.
    # TODO: aggregate loss and correct predictions over the entire loader.
    # TODO: return sample-weighted mean loss and accuracy.
    raise NotImplementedError("Complete evaluate")


#### Training and final test run (provided)

Run 5 epochs with the prescribed settings. Report each epoch's training loss/accuracy, initial/final full-training-set loss, and final test loss/accuracy. Per-epoch metrics aggregate predictions made during parameter updates; the final full-training-set metrics evaluate one fixed trained model.

A decreasing full-training-set loss is a diagnostic. The assignment does not require an exact numerical test accuracy.


In [ ]:
# Reinitialize so rerunning this cell starts the same experiment.
torch.manual_seed(2026)
model = MLP()
train_loader.generator.manual_seed(2026)
loss_fn, optimizer = make_training_components(model)
assert isinstance(loss_fn, nn.CrossEntropyLoss)
assert isinstance(optimizer, torch.optim.SGD)
assert all(group["lr"] == 0.1 for group in optimizer.param_groups)
train_eval_loader = DataLoader(train_loader.dataset, batch_size=64,
                               shuffle=False, num_workers=0)
initial_train_loss, initial_train_accuracy = evaluate(model, train_eval_loader, loss_fn)
history = []
for epoch in range(5):
    train_loss, train_accuracy = train_one_epoch(model, train_loader, loss_fn, optimizer)
    assert torch.isfinite(torch.tensor(train_loss)) and 0 <= train_accuracy <= 1
    history.append({"epoch": epoch + 1, "loss": train_loss, "accuracy": train_accuracy})
    print(f"Epoch {epoch + 1}/5: train loss={train_loss:.6f}, accuracy={train_accuracy:.2%}")
final_train_loss, final_train_accuracy = evaluate(model, train_eval_loader, loss_fn)
test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)
assert len(history) == 5 and final_train_loss < initial_train_loss
assert torch.isfinite(torch.tensor(test_loss)) and 0 <= test_accuracy <= 1
print(f"Full training set: initial loss={initial_train_loss:.6f}, final loss={final_train_loss:.6f}")
print(f"Full training set: final accuracy={final_train_accuracy:.2%}")
print(f"Final test set: loss={test_loss:.6f}, accuracy={test_accuracy:.2%}")


### 4(e). Saving and reloading (0.50)

Save `model.state_dict()` to `mnist_mlp.pt`. Create a new `MLP`, load the weights with `map_location="cpu"` and explicit `weights_only=True`, switch to evaluation mode and return the restored model.


In [ ]:
def save_and_reload(model, path="mnist_mlp.pt"):
    # TODO: save weights, construct a new MLP, load on CPU and call eval().
    raise NotImplementedError("Complete save_and_reload")


In [ ]:
restored = save_and_reload(model)
assert restored is not model and not restored.training
probe = images[:16].clone()  # Fixed images retained from the data self-check.
model.eval()
with torch.no_grad():
    expected, actual = model(probe), restored(probe)
torch.testing.assert_close(actual, expected)
print("Predicted digits:", actual.argmax(dim=1).tolist())
print("Reload check passed: logits match on the same fixed image batch.")
